### Plot the dynamic spectra from the several spacecrafts and ground stations (PSP, SolO, Wind, STEREO, LOFAR) 

# import sys
sys.path.insert(1, '../') # make sure to use the code in this repo
import glob
import json
import os
os.environ['CDF_LIB'] = '/home/peijin/cdf/cdf38_0-dist/lib'
from spacepy import pycdf
import requests
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import astropy.io.fits as fits
import scipy
from matplotlib.dates import DateFormatter
myFmt_date = DateFormatter('%Y/%m/%d')
myFmt_time = DateFormatter('%H:%M')
import matplotlib.dates as mdates
import pytz
import matplotlib.dates as dates
# try to use the precise epoch
import matplotlib as mpl
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except:
    pass
import pyspedas
from pytplot import tplot
from pytplot import options
from pytplot import get_data
import detectRadioburst_ver2 as drb
import radioTools_ver2 as rt
from skimage.transform import probabilistic_hough_line
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

In [3]:
YEAR = '2019'
MONTH = '04'
DAY = '03'
SASID_LBA = 'L700169'
SASID_HBA = ''

In [4]:
psp_hfr_dir = '/home/mnedal/ASTRON/PSP/DATA/hfr/'
psp_lfr_dir = '/home/mnedal/ASTRON/PSP/DATA/lfr/'
solo_dir = '/home/mnedal/ASTRON/SolO/DATA/'
lofar_dir = '/home/mnedal/ASTRON/LOFAR/DATA/'
wind_dir = '/home/mnedal/ASTRON/Wind/DATA/'
stereo_dir = '/home/mnedal/ASTRON/STEREO/DATA/'

In [5]:
#time_range = ['{}-{}-{}'.format(YEAR,MONTH,DAY), '{}-{}-{}'.format(YEAR,MONTH,DAY)]
time_range = [f'{YEAR}-{MONTH}-{DAY}', f'{YEAR}-{MONTH}-{DAY}']

### Wind/WAVES 

In [5]:
wind_vars = pyspedas.wind.waves(trange=time_range)

09-Nov-22 13:16:00: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/wind/waves/wav_h1/2019/
09-Nov-22 13:16:01: File is current: wind_data/waves/wav_h1/2019/wi_h1_wav_20190403_v01.cdf


In [ ]:
tplot(['E_VOLTAGE_RAD2', 'E_VOLTAGE_RAD1', 'E_VOLTAGE_TNR'], save_png=wind_dir+'wind_{}{}{}.png'.format(YEAR,MONTH,DAY))

In [7]:
RAD2_times, RAD2_int, RAD2_freq = get_data('E_VOLTAGE_RAD2')
RAD1_times, RAD1_int, RAD1_freq = get_data('E_VOLTAGE_RAD1')
TNR_times, TNR_int, TNR_freq = get_data('E_VOLTAGE_TNR')

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
plt.plot(TNR_freq)
plt.title('TNR_freq')
plt.subplot(1, 2, 2)
plt.plot(np.log10(TNR_freq))
plt.title('log$_{10}$(TNR_freq)')
plt.show()

In [ ]:
fig = plt.figure(figsize=(7,4), dpi=100)
ax = plt.gca()
im = ax.imshow(TNR_int.T, aspect='auto', origin='lower', 
            vmin=(np.mean(TNR_int)-2 * np.std(TNR_int)), 
            vmax=(np.mean(TNR_int)+3 * np.std(TNR_int)),
            extent=[dates.date2num(datetime.datetime.strptime(pyspedas.time_string(TNR_times)[0], '%Y-%m-%d %H:%M:%S.%f')), 
                    dates.date2num(datetime.datetime.strptime(pyspedas.time_string(TNR_times)[-1], '%Y-%m-%d %H:%M:%S.%f')), 
                    np.log10(TNR_freq[0]), np.log10(TNR_freq[-1])], 
            cmap='RdBu')

#fig.colorbar(im, pad=0.01)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('log$_{10}$(Frequency)')
ax.set_title('Wind/WAVES TNR: %i - %i kHz' %(TNR_freq[0], TNR_freq[-1]))
plt.show()

In [ ]:
fig = plt.figure(figsize=(7,4), dpi=100)
ax = plt.gca()
im = ax.imshow(RAD1_int.T, aspect='auto', origin='lower', 
            vmin=(np.mean(RAD1_int)-2 * np.std(RAD1_int)), 
            vmax=(np.mean(RAD1_int)+3 * np.std(RAD1_int)),
            extent=[dates.date2num(datetime.datetime.strptime(pyspedas.time_string(RAD1_times)[0], '%Y-%m-%d %H:%M:%S.%f')), 
                    dates.date2num(datetime.datetime.strptime(pyspedas.time_string(RAD1_times)[-1], '%Y-%m-%d %H:%M:%S.%f')), 
                    np.log10(RAD1_freq[0]), np.log10(RAD1_freq[-1])], 
            cmap='RdBu')

#fig.colorbar(im, pad=0.01)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('log$_{10}$(Frequency)')
ax.set_title('Wind/WAVES RAD1: %i - %i kHz' %(RAD1_freq[0], RAD1_freq[-1]))
plt.show()

### STEREO/SWAVES 

In [ ]:
rpw_vars = pyspedas.solo.rpw(trange=['2020-06-15', '2020-06-16'], datatype='hfr-surv')

tplot(['AVERAGE_NR', 'TEMPERATURE', 'FLUX_DENSITY1', 'FLUX_DENSITY2'])

In [91]:
solo_times, solo_int = get_data('FLUX_DENSITY2')

In [ ]:
plt.plot(solo_times[0:500], ".")

In [ ]:
plt.imshow(np.reshape(solo_int, [192,-1]))

In [97]:
np.arange(499)[np.diff(solo_times[0:500])>10]

array([191, 383])

In [ ]:
plt.plot(solo_int[0:500])

In [ ]:
# get the data file 
try:
    URL = 'https://spdf.gsfc.nasa.gov/pub/data/stereo/combined/swaves/level2_cdf/'+YEAR+'/stereo_level2_swaves_'+YEAR+MONTH+DAY+'_v01.cdf'
    response = requests.get(URL)
    open(os.path.join(stereo_dir, '/stereo_level2_swaves_'+YEAR+MONTH+DAY+'_v01.cdf'), 'wb').write(response.content)
except:
    pass

In [7]:
cdf_stereo = pycdf.CDF('/home/mnedal/ASTRON/STEREO/DATA/stereo_level2_swaves_20190403_v01.cdf')

In [ ]:
# read the data values
time_ste = np.array(cdf_stereo.get('Epoch'))
freq_ste = np.array(cdf_stereo.get('frequency'))/1e3
data_ste_A = np.array(cdf_stereo.get('avg_intens_ahead'))

# subtract the Mean value 
# calculate the mean intensity in each row (freq channel) 
df_ste_A = pd.DataFrame(data_ste_A.T)
df_ste_mean = df_ste_A.mean(axis=1)
# subtract that mean value from each corresponding row 
data_ste_A = df_ste_A.sub(df_ste_mean, axis=0)
#data_ste_A = pd.DataFrame(data_ste_A.T)

plt.figure(figsize=[10,4])
plt.pcolormesh(time_ste, freq_ste, data_ste_A, 
               vmin=(np.mean(data_ste_A.values)-2 * np.std(data_ste_A.values)), 
               vmax=(np.mean(data_ste_A.values)+3 * np.std(data_ste_A.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('STEREO/SWAVES: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
#plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=[10,7])
plt.pcolormesh(time_ste, freq_ste, data_ste_A, 
               vmin=(np.mean(data_ste_A.values)-2 * np.std(data_ste_A.values)), 
               vmax=(np.mean(data_ste_A.values)+3 * np.std(data_ste_A.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('STEREO/SWAVES: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.xlim(left=pd.Timestamp('2019-04-03 12:15'), right=pd.Timestamp('2019-04-03 12:50'))
#plt.yscale('log')
#plt.tight_layout()
plt.show()

In [28]:
# const background removal and gaussian smooth 
dyspec = df_ste_A.copy()
gauss_sigma = 1.5
steA_new_tmp = dyspec - np.tile(np.mean(dyspec,0), (dyspec.shape[0],1))    
steA_new = scipy.ndimage.gaussian_filter(steA_new_tmp, gauss_sigma, order=0, output=None, cval=0.0, truncate=5.0, mode='nearest')

In [ ]:
plt.figure(figsize=(10,4), dpi=100)
plt.imshow(steA_new, aspect='auto', origin='lower', 
            vmin=(np.mean(steA_new)-2 * np.std(steA_new)), 
            vmax=(np.mean(steA_new)+3 * np.std(steA_new)), 
            extent=[dates.date2num(time_ste[0]), dates.date2num(time_ste[-1]), freq_ste[0], freq_ste[-1]], 
            cmap='inferno')
#plt.pcolormesh(time_ste, freq_ste, steA_new.T, 
#            vmin=(np.mean(steA_new)-2 * np.std(steA_new)), 
#            vmax=(np.mean(steA_new)+3 * np.std(steA_new)), 
#            cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('STEREO/SWAVES: {}-{}-{} (Gaussian filter $\sigma$ = {})'.format(YEAR, MONTH, DAY, gauss_sigma))
plt.gca().xaxis.set_major_formatter(myFmt_time)
#plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=[10,7])
plt.pcolormesh(time_ste, freq_ste, steA_new, 
            vmin=(np.mean(steA_new)-2 * np.std(steA_new)), 
            vmax=(np.mean(steA_new)+3 * np.std(steA_new)), 
            cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('STEREO/SWAVES: {}-{}-{} (Gaussian filter $\sigma$ = {})'.format(YEAR, MONTH, DAY, gauss_sigma))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.xlim(left=pd.Timestamp('2019-04-03 12:15'), right=pd.Timestamp('2019-04-03 12:50'))
#plt.tight_layout()
plt.show()

----------- 

### PSP/FIELD 

# LFR 
try:
    URL = 'http://research.ssl.berkeley.edu/data/psp/data/sci/fields/l2/rfs_'+MODE+'/'+YEAR+'/'+ MONTH+'/psp_fld_l2_rfs_lfr_'+YEAR+MONTH+DAY+'_v02.cdf'
    response = requests.get(URL)
    open(os.path.join(psp_hfr_dir, '/psp_fld_l2_rfs_lfr_'+YEAR+MONTH+DAY+'_v02.cdf'), 'wb').write(response.content)
except:
     pass

# HFR 
try:
    URL = 'http://research.ssl.berkeley.edu/data/psp/data/sci/fields/l2/rfs_'+MODE+'/'+YEAR+'/'+ MONTH+'/psp_fld_l2_rfs_hfr_'+YEAR+MONTH+DAY+'_v02.cdf'
    response = requests.get(URL)
    open(os.path.join(psp_hfr_dir, '/psp_fld_l2_rfs_hfr_'+YEAR+MONTH+DAY+'_v02.cdf'), 'wb').write(response.content)
except:
     pass

In [132]:
pyspedas.psp.fields(datatype='rfs_hfr',
                    varnames='psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2',
                    trange=time_range,
                    time_clip=True)

pyspedas.psp.fields(datatype='rfs_lfr',
                    varnames='psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2',
                    trange=time_range,
                    time_clip=True)

options('psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2', 'ylog', False)
options('psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2', 'zlog', True)
options('psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2', 'ylog', False)
options('psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2', 'zlog', True)

11-Jun-22 13:40:25: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/psp/fields/l2/rfs_hfr/2019/
11-Jun-22 13:40:26: File is current: psp_data/fields/l2/rfs_hfr/2019/psp_fld_l2_rfs_hfr_20190403_v02.cdf
11-Jun-22 13:40:26: Downloading remote index: https://spdf.gsfc.nasa.gov/pub/data/psp/fields/l2/rfs_lfr/2019/


Time clip returns empty data.


11-Jun-22 13:40:27: File is current: psp_data/fields/l2/rfs_lfr/2019/psp_fld_l2_rfs_lfr_20190403_v02.cdf


Time clip returns empty data.


In [ ]:
tplot(['psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2', 'psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2'])

In [49]:
cdf_psp_hfr = pycdf.CDF(os.path.join(psp_hfr_dir, 'psp_fld_l2_rfs_hfr_'+YEAR+MONTH+DAY+'_v02.cdf'))
cdf_psp_lfr = pycdf.CDF(os.path.join(psp_lfr_dir, 'psp_fld_l2_rfs_lfr_'+YEAR+MONTH+DAY+'_v02.cdf'))

tmin_lfr = cdf_psp_lfr['epoch_lfr'].meta['SCALEMIN']
tmax_lfr = cdf_psp_lfr['epoch_lfr'].meta['SCALEMAX']

# convert pixels values to dB, # z-axis 
arr_lfr = np.array(cdf_psp_lfr.get('psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2'))
# the min power scaled power spectral density (PSD) of 1e-16 is used as a threshold according to Pulupa et al. 2020, https://doi.org/10.3847/1538-4365/ab5dc0 
# more info: https://en.wikipedia.org/wiki/Decibel 
Lp_lfr = 10 * np.log10(arr_lfr/10**-16)
# x-axis 
tm_lfr = np.array(cdf_psp_lfr.get('epoch_lfr'))
# y-axis 
freq_lfr = np.array(cdf_psp_lfr.get('frequency_lfr_auto_averages_ch0_V1V2'))/10**6

tmin_hfr = cdf_psp_hfr['epoch_hfr'].meta['SCALEMIN']
tmax_hfr = cdf_psp_hfr['epoch_hfr'].meta['SCALEMAX']

# convert pixels values to dB, # z-axis 
arr_hfr = np.array(cdf_psp_hfr.get('psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2'))
Lp_hfr = 10 * np.log10(arr_hfr/10**-16)
# x-axis 
tm_hfr = np.array(cdf_psp_hfr.get('epoch_hfr'))
# y-axis 
freq_hfr = np.array(cdf_psp_hfr.get('frequency_hfr_auto_averages_ch0_V1V2'))/10**6

# clean the dyspec by subtracting the Mean intensity from each freq channel
df_psp_hfr = pd.DataFrame(Lp_hfr.T)
df_psp_lfr = pd.DataFrame(Lp_lfr.T)

df_psp_hfr_mean = df_psp_hfr.mean(axis=1)
df_psp_lfr_mean = df_psp_lfr.mean(axis=1)

# subtract that mean value from each corresponding row 
df_psp_hfr = df_psp_hfr.sub(df_psp_hfr_mean, axis=0)
df_psp_lfr = df_psp_lfr.sub(df_psp_hfr_mean, axis=0)

In [ ]:
# plot, after cleaning 
plt.figure(figsize=(10,4), dpi=100)
plt.pcolormesh(tm_hfr, freq_hfr[1], df_psp_hfr, 
               vmin=(np.mean(df_psp_hfr.values)-2 * np.std(df_psp_hfr.values)), 
               vmax=(np.mean(df_psp_hfr.values)+3 * np.std(df_psp_hfr.values)), 
               cmap='inferno')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/FIELDS - HFR: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4), dpi=100)
plt.pcolormesh(tm_lfr, freq_lfr[1], df_psp_lfr, 
               vmin=(np.mean(df_psp_lfr.values)-2 * np.std(df_psp_lfr.values)), 
               vmax=(np.mean(df_psp_lfr.values)+3 * np.std(df_psp_lfr.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/FIELDS - LFR: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [51]:
# concat the 2 arrays of both bands 
df_lfr = pd.DataFrame(df_psp_lfr)
df_lfr.insert(loc=0, column='frequency', value=freq_lfr[1])
df_lfr.set_index(['frequency'], inplace=True)

df_hfr = pd.DataFrame(df_psp_hfr)
df_hfr.insert(loc=0, column='frequency', value=freq_hfr[1])
df_hfr.set_index(['frequency'], inplace=True)

# drop the overlapped rows, take only the first row of the duplicated group 
df_psp = pd.concat([df_lfr, df_hfr])
df_psp = df_psp.sort_index(axis=0)
df_psp = df_psp[~df_psp.index.duplicated(keep='first')]

In [ ]:
plt.figure(figsize=[10,7])
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, 
               vmin=(np.mean(df_psp.values)-2 * np.std(df_psp.values)), 
               vmax=(np.mean(df_psp.values)+3 * np.std(df_psp.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/FIELDS: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.xlim(left=pd.Timestamp('2019-04-03 12:10'), right=pd.Timestamp('2019-04-03 12:45'))
#plt.yscale('log')
#plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=[10,5])
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, 
               vmin=(np.mean(df_psp.values)-2 * np.std(df_psp.values)), 
               vmax=(np.mean(df_psp.values)+3 * np.std(df_psp.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/FIELDS: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
# plt.savefig('full_spectrum_{}.png'.format(str(tm_hfr[0].replace(microsecond=0))[:10]), format='png')
plt.show()

In [42]:
# const background removal and gaussian smooth 
dyspec_psp = df_psp.copy()
gauss_sigma = 1.5
psp_new_tmp = dyspec_psp - np.tile(np.mean(dyspec_psp,0), (dyspec_psp.shape[0],1))    
psp_new = scipy.ndimage.gaussian_filter(psp_new_tmp, gauss_sigma, order=0, output=None, cval=0.0, truncate=5.0, mode='nearest')

In [ ]:
plt.figure(figsize=[10,5])
plt.pcolormesh(tm_lfr, df_psp.index, psp_new, 
               vmin=(np.mean(psp_new)-2 * np.std(psp_new)), 
               vmax=(np.mean(psp_new)+3 * np.std(psp_new)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/FIELDS {}/{}/{} (Gaussian filter $\sigma$ = {})'.format(YEAR, MONTH, DAY, gauss_sigma))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

### LOFAR 

In [55]:
# === LBA === 
# import fits files 
lofar_LBA_fits = glob.glob(lofar_dir + '{}_{}{}{}_LBA'.format(SASID_LBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_*.fits'.format(YEAR, MONTH, DAY))
try:
    lofar_LBA_fits.remove(lofar_dir + '{}_{}{}{}_LBA'.format(SASID_LBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_LBA_OUTER.fits'.format(YEAR, MONTH, DAY))
except:
    pass
lofar_LBA_fits.sort()

# import json files 
lofar_LBA_json = glob.glob(lofar_dir + '{}_{}{}{}_LBA'.format(SASID_LBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_*.json'.format(YEAR, MONTH, DAY))
try:
    lofar_LBA_json.remove(lofar_dir + '{}_{}{}{}_LBA'.format(SASID_LBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_LBA_OUTER.json'.format(YEAR, MONTH, DAY))
except:
    pass
lofar_LBA_json.sort()

LBA_json = []
for file in lofar_LBA_json:
    f = open(file)
    LBA_json.append(json.load(f))
    f.close()

In [56]:
lofar_LBA_fits

['/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_114200_LBA_OUTER.fits',
 '/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_121200_LBA_OUTER.fits',
 '/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_122700_LBA_OUTER.fits',
 '/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_124200_LBA_OUTER.fits',
 '/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_125700_LBA_OUTER.fits',
 '/home/mnedal/ASTRON/LOFAR/DATA/L700169_20190403_LBA/LOFAR_20190403_131200_LBA_OUTER.fits']

In [57]:
LBA_list = []
# define the FREQ axis 
LBA_freq = fits.open(lofar_LBA_fits[0])[1].data['FREQ']

for i in range(len(lofar_LBA_fits)):
    # read the file 
    tmp = fits.open(lofar_LBA_fits[i])
    df = pd.DataFrame(tmp[0].data)
    # insert the datetimes as index 
    df.insert(loc=0, column='DateTime', value=tmp[2].data['TIME'])
    df.set_index(['DateTime'], inplace=True)
    # store the spectra 
    LBA_list.append(df)

In [58]:
# Concat. the list based on time index 
df_concat_LBA = pd.concat(LBA_list, axis=0)

In [ ]:
# plot 
plt.figure(figsize=(6,4), dpi=100)
plt.pcolormesh(df_concat_LBA.index, LBA_freq, df_concat_LBA.values.T, 
            vmin=(np.mean(df_concat_LBA.values)-2 * np.std(df_concat_LBA.values)),
            vmax=(np.mean(df_concat_LBA.values)+3 * np.std(df_concat_LBA.values)),
            cmap='inferno')

plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title(fits.open(lofar_LBA_fits[0])[0].header['CONTENT'])
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [60]:
# const background removal and gaussian smooth 
dyspec_lofar_lba = df_concat_LBA.copy()
gauss_sigma = 1.5
lofar_lba_new_tmp = dyspec_lofar_lba - np.tile(np.mean(dyspec_lofar_lba,0), (dyspec_lofar_lba.shape[0],1))    
lofar_lba_new = scipy.ndimage.gaussian_filter(lofar_lba_new_tmp, gauss_sigma, order=0, output=None, cval=0.0, truncate=5.0, mode='nearest')

In [ ]:
plt.figure(figsize=(6,4), dpi=100)
plt.pcolormesh(df_concat_LBA.index, LBA_freq, lofar_lba_new.T, 
            vmin=(np.mean(lofar_lba_new)-2 * np.std(lofar_lba_new)),
            vmax=(np.mean(lofar_lba_new)+3 * np.std(lofar_lba_new)),
            cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title(fits.open(lofar_LBA_fits[0])[0].header['CONTENT'])
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=[6,4])
plt.pcolormesh(df_concat_LBA.index, LBA_freq, lofar_lba_new.T, 
            vmin=(np.mean(lofar_lba_new)-2 * np.std(lofar_lba_new)),
            vmax=(np.mean(lofar_lba_new)+3 * np.std(lofar_lba_new)),
            cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title(fits.open(lofar_LBA_fits[0])[0].header['CONTENT'])
#plt.xlim(left=0.51+1.7989e4)
plt.xlim(left=pd.Timestamp('2019-04-03 12:15'), right=pd.Timestamp('2019-04-03 12:45'))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [67]:
df_concat_LBA.index = dates.num2date(df_concat_LBA.index)

In [68]:
def nearest(items, pivot):
    ''' 
    This function returns the datetime in items which is the closest to the date pivot. 
    '''
    found = min(items, key=lambda x: abs(x - pivot))
    return found

In [69]:
start_pivot = datetime.datetime(2019, 4, 3, 12, 10, tzinfo=df_concat_LBA.index[0].tz)
end_pivot = datetime.datetime(2019, 4, 3, 12, 50, tzinfo=df_concat_LBA.index[0].tz)

In [70]:
st_lof_lba = nearest(df_concat_LBA.index, start_pivot)
et_lof_lba = nearest(df_concat_LBA.index, end_pivot)

In [71]:
st_lof_lba, et_lof_lba

(Timestamp('2019-04-03 12:12:00+0000', tz='UTC'),
 Timestamp('2019-04-03 12:50:00.201342+0000', tz='UTC'))

In [72]:
st_idx_lofar_lba = np.where(df_concat_LBA.index==st_lof_lba)[0][0]
et_idx_lofar_lba = np.where(df_concat_LBA.index==et_lof_lba)[0][0]

In [73]:
st_idx_lofar_lba, et_idx_lofar_lba

(895, 3162)

In [ ]:
plt.figure(figsize=(6,4), dpi=100)
plt.pcolormesh(df_concat_LBA.index[st_idx_lofar_lba:et_idx_lofar_lba], 
            LBA_freq, 
            lofar_lba_new[st_idx_lofar_lba:et_idx_lofar_lba].T, 
            vmin=(np.mean(lofar_lba_new[st_idx_lofar_lba:et_idx_lofar_lba])-2 * np.std(lofar_lba_new[st_idx_lofar_lba:et_idx_lofar_lba])), 
            vmax=(np.mean(lofar_lba_new[st_idx_lofar_lba:et_idx_lofar_lba])+3 * np.std(lofar_lba_new[st_idx_lofar_lba:et_idx_lofar_lba])), 
            cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title(fits.open(lofar_LBA_fits[0])[0].header['CONTENT'])
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(LBA_freq)

In [ ]:
# set the same time zone as LOFAR for the PSP datetimes as well 
try:
    tm_lfr = [pytz.timezone('utc').localize(i) for i in tm_lfr]
except:
    pass

In [ ]:
st_psp = nearest(tm_lfr, start_pivot)
et_psp = nearest(tm_lfr, end_pivot)

In [ ]:
st_psp, et_psp

In [ ]:
st_idx_psp = next((i for i, j in enumerate(tm_lfr) if j == st_psp), None)
et_idx_psp = next((i for i, j in enumerate(tm_lfr) if j == et_psp), None)

In [ ]:
st_idx_psp, et_idx_psp

In [ ]:
plt.figure(figsize=(10,6), dpi=100)

plt.pcolor(df_concat_LBA.index, LBA_freq, lofar_lba_new.T, 
          vmin=(np.mean(lofar_lba_new)-2 * np.std(lofar_lba_new)), 
          vmax=(np.mean(lofar_lba_new)+3 * np.std(lofar_lba_new)), 
          cmap='inferno')

plt.pcolor(tm_lfr, df_psp.index, df_psp.values, 
               vmin=(np.mean(df_psp.values)-2 * np.std(df_psp.values)), 
               vmax=(np.mean(df_psp.values)+3 * np.std(df_psp.values)), 
               cmap='inferno')

plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('{}/{}/{}'.format(YEAR, MONTH, DAY))
plt.xlim(left=st_psp, right=et_psp)
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
# check cadence 
lof_cadence = round((df_concat_LBA.index[1] - df_concat_LBA.index[0]).seconds + (df_concat_LBA.index[1] - df_concat_LBA.index[0]).microseconds/1e6)
psp_cadence = round((tm_lfr[1] - tm_lfr[0]).seconds + (tm_lfr[1] - tm_lfr[0]).microseconds/1e6)
lof_cadence, psp_cadence

In [ ]:
# resample LOFAR wrt PSP 
df_lof_lba = pd.DataFrame(lofar_lba_new)
df_lof_lba.insert(loc=0, column='DateTime', value=df_concat_LBA.index)
df_lof_lba.set_index(['DateTime'], inplace=True)

resamp_lofar_lba = df_lof_lba.resample(str(psp_cadence)+'S').sum()

In [ ]:
resamp_lofar_lba.index[1] - resamp_lofar_lba.index[0]

In [ ]:
tm_lfr[1] - tm_lfr[0]

In [ ]:
psp_data = df_psp.values
psp_freq = df_psp.index.values
psp_time = tm_lfr

In [ ]:
psp_struct = pd.DataFrame(psp_data.T)
psp_struct.insert(loc=0, column='DateTime', value=tm_lfr)
psp_struct.set_index(['DateTime'], inplace=True)

In [ ]:
st_psp, et_psp

In [ ]:
''' 
Time travel for radio waves
----------------------------
For PSP: 86.54 light seconds, or 1.44 light minutes.
For Earth: 498.91 light seconds, or 8.32 light minutes. 
'''

In [ ]:
# from psp to lofar 
498.91 - 86.54

In [ ]:
round((498.91 - 86.54)/psp_cadence)

In [ ]:
# shift PSP wrt LOFAR 
psp_shift = psp_struct.shift(periods=59, fill_value=0)

In [ ]:
# After resampling and shifting 
plt.figure(figsize=(7,5), dpi=100)

plt.pcolor(resamp_lofar_lba.index, LBA_freq, resamp_lofar_lba.values.T, 
            vmin=(np.mean(resamp_lofar_lba.values)-2 * np.std(resamp_lofar_lba.values)), 
            vmax=(np.mean(resamp_lofar_lba.values)+3 * np.std(resamp_lofar_lba.values)), 
            cmap='inferno')

plt.pcolor(psp_shift.index, psp_freq, psp_shift.values.T, 
            vmin=(np.mean(psp_shift.values)-2 * np.std(psp_shift.values)), 
            vmax=(np.mean(psp_shift.values)+3 * np.std(psp_shift.values)), 
            cmap='inferno')

plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('{}/{}/{}'.format(YEAR, MONTH, DAY))
plt.xlim(left=st_psp, right=et_psp)
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
''' 
1. concat. lofar with psp data 
2. norm. the int. vals. --> calculate (I_psp/I_LOFAR) at 20MHz, and multiply the ratio to LOFAR 
3. plot the log-scale 
'''
# resamp_lofar_lba.index, LBA_freq, resamp_lofar_lba.values
# psp_shift.index, psp_freq, psp_shift.values

In [ ]:
plt.figure(figsize=(6,4), dpi=100)

plt.pcolor(resamp_lofar_lba.index, LBA_freq, resamp_lofar_lba.values.T, 
            vmin=(np.mean(resamp_lofar_lba.values)-2 * np.std(resamp_lofar_lba.values)), 
            vmax=(np.mean(resamp_lofar_lba.values)+3 * np.std(resamp_lofar_lba.values)), 
            cmap='inferno')

In [ ]:
sub1_tm_lof = nearest(resamp_lofar_lba.index, start_pivot)
sub2_tm_lof = nearest(resamp_lofar_lba.index, end_pivot)

In [ ]:
subset1_lof = resamp_lofar_lba.index.get_loc(sub1_tm_lof)
subset2_lof = resamp_lofar_lba.index.get_loc(sub2_tm_lof)

In [ ]:
# After resampling and shifting 
plt.figure(figsize=(7,5), dpi=100)

plt.pcolor(resamp_lofar_lba.index[subset1_lof:subset2_lof], LBA_freq, resamp_lofar_lba.values[subset1_lof:subset2_lof].T, 
            vmin=(np.mean(resamp_lofar_lba.values[subset1_lof:subset2_lof])-2 * np.std(resamp_lofar_lba.values[subset1_lof:subset2_lof])), 
            vmax=(np.mean(resamp_lofar_lba.values[subset1_lof:subset2_lof])+3 * np.std(resamp_lofar_lba.values[subset1_lof:subset2_lof])), 
            cmap='inferno')

plt.pcolor(psp_shift[st_idx_psp:et_idx_psp].index, 
            psp_freq, 
            psp_shift[st_idx_psp:et_idx_psp].values.T, 
            vmin=(np.mean(psp_shift.values)-2 * np.std(psp_shift.values)), 
            vmax=(np.mean(psp_shift.values)+3 * np.std(psp_shift.values)), 
            cmap='inferno')

plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('{}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
# Normalizing LOFAR data to match PSP 
idx_lof_20MHz = np.where(LBA_freq==nearest(LBA_freq, 20))[0][0]
idx_psp_20MHz = np.where(psp_freq==nearest(psp_freq, 20))[0][0]

I_LOFAR = resamp_lofar_lba.values[subset1_lof:subset2_lof, idx_lof_20MHz]
I_psp = psp_shift.values[st_idx_psp:et_idx_psp, idx_psp_20MHz]

In [ ]:
plt.plot(I_LOFAR, label='LOFAR at {:.2f} MHz'.format(nearest(LBA_freq, 20)))
plt.plot(I_psp, label='PSP at {:.2f} MHz'.format(nearest(psp_freq, 20)))
plt.legend(loc='best', frameon=False)
plt.xlabel('Timestep')
plt.ylabel('Intensity')
plt.title('Nearest frequency channels to 20 MHz in PSP and LOFAR')
plt.grid(alpha=0.5)
plt.show()

In [ ]:
# Normalize LOFAR 
norm_lof_lba = (I_psp/I_LOFAR)*resamp_lofar_lba.values[subset1_lof:subset2_lof].T
# replace -inf with zeros  
norm_lof_lba = pd.DataFrame(norm_lof_lba).replace(-np.inf, 0)

fig = plt.figure(figsize=(7,6), dpi=100)
ax1 = plt.subplot(2, 1, 1)
dyspec_lof = ax1.pcolor(resamp_lofar_lba.index[subset1_lof:subset2_lof], LBA_freq, norm_lof_lba, 
             #           vmin=(np.mean(norm_lof_lba.values)-2 * np.std(norm_lof_lba.values)), 
             #           vmax=(np.mean(norm_lof_lba.values)+3 * np.std(norm_lof_lba.values)), 
                        cmap='inferno')
cb1 = plt.colorbar(dyspec_lof)
ax1.set_ylabel('Frequency (MHz)')

ax2 = plt.subplot(2, 1, 2)
dyspec_psp = ax2.pcolor(psp_shift[st_idx_psp:et_idx_psp].index, 
                        psp_freq, 
                        psp_shift[st_idx_psp:et_idx_psp].values.T, 
              #          vmin=(np.mean(psp_shift.values)-2 * np.std(psp_shift.values)), 
              #          vmax=(np.mean(psp_shift.values)+3 * np.std(psp_shift.values)), 
                        cmap='inferno')
cb2 = plt.colorbar(dyspec_psp)
ax2.set_ylabel('Frequency (MHz)')

ax2.set_xlabel('Time (UT)')
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
# Normalize PSP 
norm_psp = (I_psp/I_LOFAR)*psp_shift[st_idx_psp:et_idx_psp].values.T
# replace -inf with zeros  
norm_psp = pd.DataFrame(norm_psp).replace(-np.inf, 0)

fig = plt.figure(figsize=(7,6), dpi=100)
ax1 = plt.subplot(2, 1, 1)
dyspec_lof = ax1.pcolor(resamp_lofar_lba.index[subset1_lof:subset2_lof], LBA_freq, resamp_lofar_lba.values[subset1_lof:subset2_lof].T, 
             #           vmin=(np.mean(resamp_lofar_lba.values[subset1_lof:subset2_lof])-2 * np.std(resamp_lofar_lba.values[subset1_lof:subset2_lof])), 
             #           vmax=(np.mean(resamp_lofar_lba.values[subset1_lof:subset2_lof])+3 * np.std(resamp_lofar_lba.values[subset1_lof:subset2_lof])), 
                        cmap='inferno')
cb1 = plt.colorbar(dyspec_lof)
ax1.set_ylabel('Frequency (MHz)')

ax2 = plt.subplot(2, 1, 2)
dyspec_psp = ax2.pcolor(psp_shift[st_idx_psp:et_idx_psp].index, 
                        psp_freq, 
                        norm_psp, 
              #          vmin=(np.mean(psp_shift.values)-2 * np.std(psp_shift.values)), 
              #          vmax=(np.mean(psp_shift.values)+3 * np.std(psp_shift.values)), 
                        cmap='inferno')
cb2 = plt.colorbar(dyspec_psp)
ax2.set_ylabel('Frequency (MHz)')

ax2.set_xlabel('Time (UT)')
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
### === LOFAR LBA ======= 
# define dataframe 
df_LOFAR_LBA = pd.DataFrame(resamp_lofar_lba.values[subset1_lof:subset2_lof])
# define time 
df_LOFAR_LBA.insert(loc=0, column='DateTime', value=resamp_lofar_lba.index[subset1_lof:subset2_lof])
df_LOFAR_LBA.set_index(['DateTime'], inplace=True)
# define freq 
df_LOFAR_LBA.columns = LBA_freq

In [ ]:
### === PSP HFR + LFR ======= 
# define dataframe 
df_PSP = pd.DataFrame(psp_shift[st_idx_psp:et_idx_psp].values)
# define time 
df_PSP.insert(loc=0, column='DateTime', value=psp_shift[st_idx_psp:et_idx_psp].index)
df_PSP.set_index(['DateTime'], inplace=True)
# define freq 
df_PSP.columns = psp_freq

In [ ]:
total_freq = np.concatenate([df_PSP.columns, df_LOFAR_LBA.columns])
finool = np.concatenate([df_PSP.values, df_LOFAR_LBA.values], axis=1)

In [ ]:
plt.figure(figsize=(7,5), dpi=100)

plt.pcolor(df_PSP.index, total_freq, finool.T, 
            vmin=(np.mean(finool)-2 * np.std(finool)), 
            vmax=(np.mean(finool)+3 * np.std(finool)), 
            cmap='inferno')

plt.colorbar()
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('{}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
#plt.savefig(lofar_dir + '{}_{}{}{}_LBA'.format(SASID_LBA, YEAR, MONTH, DAY) + '/lofar_LBA.png', dpi=100, format='png')
plt.show()

In [ ]:
plt.figure(figsize=(7,5), dpi=100)

plt.pcolor(df_PSP.index, total_freq, finool.T, 
            vmin=(np.mean(finool)-2 * np.std(finool)), 
            vmax=(np.mean(finool)+3 * np.std(finool)), 
            cmap='inferno')

plt.colorbar()
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.yscale('log')
plt.title('{}/{}/{}'.format(YEAR, MONTH, DAY))
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.tight_layout()
plt.show()

In [ ]:
# === HBA === 
# import fits files 
lofar_HBA_fits = glob.glob(lofar_dir + '{}_{}{}{}_HBA'.format(SASID_HBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_*.fits'.format(YEAR, MONTH, DAY))
try:
    lofar_HBA_fits.remove(lofar_dir + '{}_{}{}{}_HBA'.format(SASID_HBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_HBA_OUTER.fits'.format(YEAR, MONTH, DAY))
except:
    pass
lofar_HBA_fits.sort()

# import json files 
lofar_HBA_json = glob.glob(lofar_dir + '{}_{}{}{}_HBA'.format(SASID_HBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_*.json'.format(YEAR, MONTH, DAY))
try:
    lofar_HBA_json.remove(lofar_dir + '{}_{}{}{}_HBA'.format(SASID_HBA, YEAR, MONTH, DAY) + '/LOFAR_{}{}{}_HBA_OUTER.json'.format(YEAR, MONTH, DAY))
except:
    pass
lofar_HBA_json.sort()

HBA_json = []
for file in lofar_HBA_json:
    f = open(file)
    HBA_json.append(json.load(f))
    f.close()

HBA_list = []
# define the FREQ axis 
HBA_freq = fits.open(lofar_HBA_fits[0])[1].data['FREQUENCY'][0]

for i in range(len(lofar_HBA_fits)):
    # read the file 
    tmp = fits.open(lofar_HBA_fits[i])
    # define time axis 
    HBA_time = tmp[1].data['TIME'][0]
    # clean the data by subtracting the mean 
    df = pd.DataFrame(tmp[0].data)
    # insert the datetimes as index 
    df.insert(loc=0, column='DateTime', value=HBA_time)
    df.set_index(['DateTime'], inplace=True)
    # store the spectra 
    HBA_list.append(df)

# Concat. the list based on time index 
df_concat_HBA = pd.concat(HBA_list, axis=0)

# Plot 
fig = plt.figure(figsize=(12,4), dpi=100)
ax = plt.gca()
ax.imshow(df_concat_HBA.values.T, aspect='auto', origin='lower', 
          vmin=(np.mean(df_concat_HBA.values)-2*np.std(df_concat_HBA.values)),
          vmax=(np.mean(df_concat_HBA.values)+3*np.std(df_concat_HBA.values)),
          extent=[df_concat_HBA.index[0], df_concat_HBA.index[-1], HBA_freq[0], HBA_freq[-1]], 
          cmap='inferno')

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(fits.open(lofar_HBA_fits[0])[0].header['CONTENT'])
plt.savefig(lofar_dir + '{}_{}{}{}_HBA'.format(SASID_HBA, YEAR, MONTH, DAY) + '/lofar_HBA.png', dpi=100, format='png')
plt.show()

In [ ]:
tmp = fits.open(lofar_HBA_fits[i])
tmp.info()

In [ ]:
[print(fits.open(lofar_HBA_fits[i]).info(),'\n') for i in range(len(lofar_HBA_fits))]

In [ ]:
# get freq info 
first_freq = LBA_data[0]['freq_range'][0]
last_freq = LBA_data[0]['freq_range'][1]
num_steps_freq = LBA_data[0]['n_freq']
freq_step = last_freq/num_steps_freq
freq_ax_LBA = np.arange(first_freq, last_freq, freq_step-0.026)

In [ ]:
def get_data_from_fits(path):
    with fits.open(path) as hdu:
        data = hdu[0].data.astype(np.float64)
    return data

In [ ]:
# ------------- HBA -------------------- 

# Returns JSON object as a dictionary 
HBA_data = []
for file in HBA_paths_json:
    f = open(file)
    HBA_data.append(json.load(f))
    f.close()

# get freq info 
first_freq = HBA_data[0]['freq_range'][0]
last_freq = HBA_data[0]['freq_range'][1]
num_steps_freq = HBA_data[0]['n_freq']
freq_step = last_freq/num_steps_freq
freq_ax_HBA = np.arange(first_freq, last_freq, freq_step-0.276)

In [ ]:
df_HBA_all = []

for i in range((len(HBA_paths_fits))):
    # read the file 
    fits_data = fits.open(HBA_paths_fits[i])[0]
    arr = fits_data.data.T
    # get time info 
    start_obs = HBA_data[i]['time_range'][0]
    end_obs = HBA_data[i]['time_range'][1]
    # define time axis 
    time_ax_HBA = pd.date_range(start=start_obs, end=end_obs, periods=fits_data.header['NAXIS2'])
    # clean the data by subtracting the mean 
    df = pd.DataFrame(arr)
    # calculate the mean intensity in each row (freq channel) 
    df_mean = df.mean(axis=1)
    # subtract that mean value from each corresponding row 
    df_submean = df.sub(df_mean, axis=0)
    # transpose the table 
    df_submean = df_submean.T
    # insert the datetimes as index 
    df_submean.insert(loc=0, column='DateTime', value=time_ax_HBA)
    df_submean.set_index(['DateTime'], inplace=True)
    # store the cleaned tables 
    df_HBA_all.append(df_submean)

In [ ]:
# Concat. the list based on time index 
concat_HBA_all = pd.concat(df_HBA_all, axis=0)

In [ ]:
# convert to string to remove the milliseconds 
concat_HBA_all.index = concat_HBA_all.index.strftime('%Y-%m-%d %H:%M:%S')
# convert it back to datetime object 
concat_HBA_all.index = [datetime.datetime.strptime(tm, '%Y-%m-%d %H:%M:%S') for tm in concat_HBA_all.index]
correct_time_ax_HBA = pd.date_range(start=concat_HBA_all.index[0], end=concat_HBA_all.index[-1], freq='S')
# Drop rows with duplicate index values 
concat_HBA_all = concat_HBA_all.loc[~concat_HBA_all.index.duplicated(), :]
new_df_HBA_all = concat_HBA_all.reindex(correct_time_ax_HBA, fill_value=0)

In [ ]:
plt.figure(figsize=(6,4), dpi=100)
#ax.imshow(new_df_HBA_all.T, 
#          aspect='auto', 
#          origin='lower', 
#          vmin=(np.mean(new_df_HBA_all.values)-2 * np.std(new_df_HBA_all.values)), 
#          vmax=(np.mean(new_df_HBA_all.values)+3 * np.std(new_df_HBA_all.values)), 
          #extent=[wind_time[0], wind_time[-1], freq_MHz[0], freq_MHz[-1]], 
#          cmap='inferno')
plt.pcolor(new_df_HBA_all.index, freq_ax_HBA, new_df_HBA_all.T, 
              vmin=(np.mean(new_df_HBA_all.values)-2 * np.std(new_df_HBA_all.values)), 
              vmax=(np.mean(new_df_HBA_all.values)+3 * np.std(new_df_HBA_all.values)), 
             cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.title('LOFAR/HBA: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.tight_layout()
plt.show()

In [ ]:
# ------------- LBA -------------------- 
LBA_data = []
for file in LBA_paths_json:
    f = open(file)
    LBA_data.append(json.load(f))
    f.close()

# get freq info 
first_freq = LBA_data[0]['freq_range'][0]
last_freq = LBA_data[0]['freq_range'][1]
num_steps_freq = LBA_data[0]['n_freq']
freq_step = last_freq/num_steps_freq
freq_ax_LBA = np.arange(first_freq, last_freq, freq_step-0.026)
        
df_LBA_all = []
for i in range((len(LBA_paths_fits))):
    # read the file 
    fits_data = fits.open(LBA_paths_fits[i])[0]
    arr = fits_data.data.T
    # get time info 
    start_obs = LBA_data[i]['time_range'][0]
    end_obs = LBA_data[i]['time_range'][1]
    # define time axis 
    time_ax_LBA = pd.date_range(start=start_obs, end=end_obs, periods=fits_data.header['NAXIS2'])
    # clean the data by subtracting the mean 
    df = pd.DataFrame(arr)
    # calculate the mean intensity in each row (freq channel) 
    df_mean = df.mean(axis=1)
    # subtract that mean value from each corresponding row 
    df_submean = df.sub(df_mean, axis=0)
    # transpose the table 
    df_submean = df_submean.T
    # insert the datetimes as index 
    df_submean.insert(loc=0, column='DateTime', value=time_ax_LBA)
    df_submean.set_index(['DateTime'], inplace=True)
    # store the cleaned tables 
    df_LBA_all.append(df_submean)

# Concat. the list based on time index 
concat_LBA_all = pd.concat(df_LBA_all, axis=0)
# convert to string to remove the milliseconds 
concat_LBA_all.index = concat_LBA_all.index.strftime('%Y-%m-%d %H:%M:%S')
# convert it back to datetime object 
concat_LBA_all.index = [datetime.datetime.strptime(tm, '%Y-%m-%d %H:%M:%S') for tm in concat_LBA_all.index]

correct_time_ax_LBA = pd.date_range(start=concat_LBA_all.index[0], end=concat_LBA_all.index[-1], freq='S')
# Drop rows with duplicate index values 
concat_LBA_all = concat_LBA_all.loc[~concat_LBA_all.index.duplicated(), :]
new_df_LBA_all = concat_LBA_all.reindex(correct_time_ax_LBA, fill_value=0)

plt.figure(figsize=(6,4), dpi=100)
#ax.imshow(new_df_LBA_all.values.T, 
#          aspect='auto', 
#          origin='lower', 
#          vmin=(np.mean(new_df_LBA_all.values)-2 * np.std(new_df_LBA_all.values)), 
#          vmax=(np.mean(new_df_LBA_all.values)+3 * np.std(new_df_LBA_all.values)), 
          #extent=[wind_time[0], wind_time[-1], freq_MHz[0], freq_MHz[-1]], 
#          cmap='inferno')
plt.pcolor(new_df_LBA_all.index, freq_ax_LBA, new_df_LBA_all.values.T, 
               vmin=(np.mean(new_df_LBA_all.values)-2 * np.std(new_df_LBA_all.values)), 
               vmax=(np.mean(new_df_LBA_all.values)+3 * np.std(new_df_LBA_all.values)), 
               cmap='inferno')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt_time)
plt.title('LOFAR/LBA: {}/{}/{}'.format(YEAR, MONTH, DAY))
plt.tight_layout()
plt.show()

In [ ]:
sigma = 0.3
(lofar_hba_new_tmp, lofar_hba_new) = drb.preproc(new_df_HBA_all, gauss_sigma=sigma)
(lofar_lba_new_tmp, lofar_lba_new) = drb.preproc(new_df_LBA_all, gauss_sigma=sigma)

fig = plt.figure(figsize=(6,4), dpi=100)
ax = plt.gca()
ax.imshow(lofar_hba_new.T, 
          aspect='auto', 
          origin='lower', 
          vmin=(np.mean(lofar_hba_new)-2 * np.std(lofar_hba_new)), 
          vmax=(np.mean(lofar_hba_new)+3 * np.std(lofar_hba_new)), 
          #extent=[new_df_HBA_all.index[0], new_df_HBA_all.index[-1], freq_ax_HBA[0], freq_ax_HBA[-1]], 
          cmap='inferno')
#ax.xaxis_date()
#ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time bins')
ax.set_ylabel('Frequency bins')
plt.title('LOFAR/HBA: (Gaussian filter $\sigma$={}): {}/{}/{}'.format(sigma, YEAR, MONTH, DAY))
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(6,4), dpi=100)
ax = plt.gca()
ax.imshow(lofar_lba_new.T, 
          aspect='auto', 
          origin='lower', 
          vmin=(np.mean(lofar_lba_new)-2 * np.std(lofar_lba_new)), 
          vmax=(np.mean(lofar_lba_new)+3 * np.std(lofar_lba_new)), 
          #extent=[new_df_LBA_all.index[0], new_df_LBA_all.index[-1], freq_ax_LBA[0], freq_ax_LBA[-1]], 
          cmap='inferno')
#ax.xaxis_date()
#ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel('Time bins')
ax.set_ylabel('Frequency bins')
plt.title('LOFAR/LBA: (Gaussian filter $\sigma$={}): {}/{}/{}'.format(sigma, YEAR, MONTH, DAY))
plt.tight_layout()
plt.show()

### SolO/RPW 

In [ ]:
cdf_solo = pycdf.CDF(os.path.join(solo_dir, 'solo_L2_rpw-hfr-surv_'+YEAR+MONTH+DAY+'_V01.cdf'))

In [ ]:
tm = cdf_solo['Epoch'][:]
freq = cdf_solo['FREQUENCY'][:]
flux = cdf_solo['FLUX_DENSITY1'][:]

In [ ]:
for key in cdf_solo.keys():
    print(key)
    print(cdf_solo[key].meta)
    print('\n')